In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools
from sklearn.metrics.cluster import adjusted_mutual_info_score, adjusted_rand_score
from scipy import stats
from scipy.stats import pearsonr, kruskal, chi2_contingency, mannwhitneyu, linregress, fisher_exact
import seaborn as sns
from ptitprince import PtitPrince as pt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test
from lifelines.plotting import add_at_risk_counts
from sklearn.utils.class_weight import compute_class_weight
from pandas.api.types import is_numeric_dtype
from statsmodels.stats.multitest import multipletests
import statsmodels
from scipy.cluster.hierarchy import fcluster
import forestplot as fp
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer

In [ ]:
# Function to remove clusters containing < 10% patients
# output is also list with top x outlier patients (patients most typically found in small clusters)
def remove_small_clusters(df: pd.DataFrame, n_outliers: int, verbose: bool):
    valid_results = df[df['relative_cluster_sizes'].apply(lambda d: all(value >= 0.1 for value in d.values()))]
    outlier_results = df[df['relative_cluster_sizes'].apply(lambda f: any(value < 0.1 for value in f.values()))]
    outlier_patients = {}
    for index, row in outlier_results.iterrows():
        cluster_sizes = row['relative_cluster_sizes']
        small_clusters = [cluster for cluster, size in cluster_sizes.items() if size < 0.1]
        patients = row['y_pred_idx']
        clusters = row['y_pred']
        for patient, cluster in zip(patients, clusters):
            if cluster in small_clusters:
                if patient not in outlier_patients:
                    outlier_patients[patient] = 1
                else:
                    outlier_patients[patient] += 1
    for key in outlier_patients:
        outlier_patients[key] /= len(outlier_results)
    top_outliers = sorted(outlier_patients.items(), key=lambda x: x[1], reverse=True)[:n_outliers]
    df_top_outliers = pd.DataFrame(data=top_outliers, columns=['Patient ID', 'Count'])
    top_outlier_patients = [patient for patient, count in top_outliers]
    if verbose == True:
        print(f"Top {n_outliers} outlier patients: {top_outlier_patients}")
    return valid_results, outlier_results, df_top_outliers

In [ ]:
# Function for consensus clustering
def consensus_matrix(df: pd.DataFrame):
    patients = list(set(sum(df['y_pred_idx'], [])))
    connectivity_matrix = pd.DataFrame(0, index=patients, columns=patients)
    indicator_matrix = pd.DataFrame(0, index=patients, columns=patients)
    for index, row in df.iterrows():
        y_pred_idx = row['y_pred_idx']
        y_pred = row['y_pred']
        for i in range(len(y_pred_idx)):
            patient_i = y_pred_idx[i]
            cluster_i = y_pred[i]
            for j in range(i + 1, len(y_pred_idx)):
                patient_j = y_pred_idx[j]
                cluster_j = y_pred[j]
                indicator_matrix.loc[patient_i, patient_j] += 1
                indicator_matrix.loc[patient_j, patient_i] += 1
                if cluster_i == cluster_j:
                    connectivity_matrix.loc[patient_i, patient_j] += 1
                    connectivity_matrix.loc[patient_j, patient_i] += 1
    consensus_matrix = connectivity_matrix.div(indicator_matrix).fillna(0)
    return consensus_matrix

In [ ]:
# Function to perform clinical enrichment of labels to each row
def clinical_enrichment(results, clinical_data):
    # Sort labels
    results["sorted_y_pred_idx"] = results["y_pred_idx"].apply(sorted)
    results["sorted_y_pred"] = results.apply(
        lambda row: [row["y_pred"][row["y_pred_idx"].index(patient_id)] for patient_id in row["sorted_y_pred_idx"]],
        axis=1)
    
    # Add clinical data
    clinical_data = clinical_data[['Patient ID', 'Fraction Genome Altered', 'Diagnosis Age', 'Sex', 'Race Category', 
                                   'Adjuvant Postoperative Targeted Therapy Administered Indicator', 'Alcohol History Documented', 'Tumor resected max dimension',
                                   'American Joint Committee on Cancer Metastasis Stage Code', 'American Joint Committee on Cancer Tumor Stage Code',
                                   'Chronic Pancreatitis Personal Medical History Indicator', 'Did patient start adjuvant postoperative radiotherapy?', 
                                   'Disease Free Status', 'Family History of Cancer', 'Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code', 
                                   'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Neoplasm Histologic Grade', 'TMB (nonsynonymous)',
                                   'New Neoplasm Event Post Initial Therapy Indicator', 'Overall Survival (Months)', 'Overall Survival Status', 'Disease Free (Months)', 
                                   'Participant Personal Medical History Diabetes Mellitus Ind-3', 'Patient Primary Tumor Site', 'Prior Cancer Diagnosis Occurence', 
                                   'Surgical Margin Resection Status', 'Patient Smoking History Category', 'Person Neoplasm Status', 'Primary Therapy Outcome Success Type']]
    clinical_data.set_index('Patient ID', inplace=True)
    
    # Convert necessary data 
    clinical_data['Overall Survival Status'] = clinical_data['Overall Survival Status'].str.split(':').str[0].astype(int)
    clinical_data['Patient Smoking History Category'] = (clinical_data['Patient Smoking History Category']
                                                         .where(clinical_data['Patient Smoking History Category'].isna(), 
                                                                clinical_data['Patient Smoking History Category'].astype(float).astype(str)))
    clinical_data_columns = [col for col in clinical_data.columns if col != 'Patient ID']

    def get_filtered_clinical_data(patient_ids, clinical_data, column_name):
        filtered_data = clinical_data.loc[patient_ids]
        return filtered_data[column_name].values.tolist()
    for column in clinical_data_columns:
        results[column] = results.apply(lambda row: get_filtered_clinical_data(row['sorted_y_pred_idx'], clinical_data, column), axis=1)
    
    # Logrank test
    def calculate_logrank_pvalue(row):
        df = pd.DataFrame({
            'cluster': row['sorted_y_pred'],
            'vital_status': row['Overall Survival Status'],
            'days_to_death': row['Overall Survival (Months)']
        })
        kmf = KaplanMeierFitter()
        test_results = multivariate_logrank_test(df['days_to_death'], df['cluster'], df['vital_status'])
        return test_results.p_value
    results['pvalue_logrank'] = results.apply(calculate_logrank_pvalue, axis=1)
    
    # Function for p-values (Kruskal-Wallis, chi2)
    def pvalue_tests(row, clinical_data_columns):
        pvalues = []
        for variable in clinical_data_columns:
            df = pd.DataFrame({
                'cluster': row['y_pred'],
                variable: row[variable]
            })
            if pd.api.types.is_numeric_dtype(df[variable]):
                # Kruskal-Wallis test for numerical variables
                test_numerical = [df[df['cluster'] == cluster][variable].dropna().to_numpy() for cluster in df['cluster'].unique()]
                stat, p_value_kruskal = kruskal(*test_numerical)
                pvalues.append(p_value_kruskal)
            else:
                # Chi-square contingency test for categorical variables
                test_discrete = pd.crosstab(df['cluster'], df[variable])
                chi2, p_value_chi2, dof, freq = chi2_contingency(test_discrete)
                pvalues.append(p_value_chi2)
        reject, pvals_corr, asidack, abonf = multipletests(pvals=pvalues, alpha=0.05, method='fdr_bh')
        for idx, variable in enumerate(clinical_data_columns):
            row[f"pvalue_{variable}"] = pvals_corr[idx]
        return row
    
    clinical_data_columns = [col for col in clinical_data_columns if col not in ['Overall Survival Status', 'Overall Survival (Months)']]
    results = results.apply(lambda row: pvalue_tests(row, clinical_data_columns), axis=1)
    columns_to_check = [f'pvalue_{variable}' for variable in clinical_data_columns]
    results['n_enriched_clinical'] = (results[columns_to_check] < 0.05).sum(axis=1)
    return results

In [ ]:
# Function to calculate stability metrics
def calculate_stability_metrics(results: pd.DataFrame, random_state=None, progress_bar=True):

    alg_stability = results[['dataset', 'view_combination', 'algorithm', 'n_clusters', 'missing_percentage', 'amputation_mechanism', 'imputation', 'run_n', "sorted_y_pred", 
                             "sorted_y_pred_idx", 'silhouette', 'normalised_silhouette', 'vrc', 'db', 'dbcv', 'dunn', "dhi", "ssei", 'rsi', 'bhi', 'pvalue_logrank', 'n_enriched_clinical']]
    if alg_stability["imputation"].nunique() != 1:
        alg_stability = alg_stability.loc[
            (alg_stability["missing_percentage"] == 0) | (alg_stability["imputation"])
            ]
    alg_uns_metrics = alg_stability.drop(columns=["sorted_y_pred", "sorted_y_pred_idx",'imputation', 'run_n'])
    
    # Group by taking mean of metrics
    alg_uns_metrics = alg_uns_metrics.groupby(
        ["dataset", "algorithm", "missing_percentage", "amputation_mechanism", "view_combination", "n_clusters"], as_index=False).mean()

    iterator = alg_stability["dataset"].unique()
    if progress_bar:
        iterator = tqdm(iterator)

    for dataset in iterator:
        preds_dataset = alg_stability.loc[
            (alg_stability["dataset"] == dataset), ["missing_percentage", "algorithm", 'amputation_mechanism', 'n_clusters', 
                                                    'view_combination', "run_n", "sorted_y_pred", "sorted_y_pred_idx"]]
        for alg in preds_dataset["algorithm"].unique():
            pred_alg = preds_dataset[preds_dataset["algorithm"] == alg]
            for missing_percentage in pred_alg["missing_percentage"].unique():
                pred_missing_alg = pred_alg[pred_alg["missing_percentage"] == missing_percentage]
                for amputation_mechanism in pred_missing_alg["amputation_mechanism"].unique():
                    pred_missing_ampt_alg = pred_missing_alg[
                        pred_missing_alg["amputation_mechanism"] == amputation_mechanism]
                    
                    for view in pred_missing_ampt_alg["view_combination"].unique():
                        pred_missing_ampt_alg_view = pred_missing_ampt_alg[
                            pred_missing_ampt_alg["view_combination"] == view]
                        for cluster in pred_missing_ampt_alg_view["n_clusters"].unique():
                            pred_missing_ampt_alg_view_clus = pred_missing_ampt_alg_view[
                                pred_missing_ampt_alg_view['n_clusters'] == cluster]

                            amis, aris = [], []
                            
                            for run_1, run_2 in set(itertools.combinations(pred_missing_ampt_alg_view_clus["run_n"].unique(), 2)):
                                pred1_alg = pred_missing_ampt_alg_view_clus.loc[
                                    (pred_missing_ampt_alg_view_clus["run_n"] == run_1), "sorted_y_pred"].to_list()[0]
                                pred2_alg = pred_missing_ampt_alg_view_clus.loc[
                                    (pred_missing_ampt_alg_view_clus["run_n"] == run_2), "sorted_y_pred"].to_list()[0]
                                
                                pred1_idx = pred_missing_ampt_alg_view_clus.loc[(
                                    pred_missing_ampt_alg_view_clus["run_n"] == run_1), "sorted_y_pred_idx"].to_list()[0]
                                pred2_idx = pred_missing_ampt_alg_view_clus.loc[(
                                    pred_missing_ampt_alg_view_clus["run_n"] == run_2), "sorted_y_pred_idx"].to_list()[0]
        
                                # Only select samples in common for stability metrics
                                common_samples = list(set(pred1_idx) & set(pred2_idx))
                                pred1_common = [pred1_alg[pred1_idx.index(i)] for i in common_samples]
                                pred2_common = [pred2_alg[pred2_idx.index(i)] for i in common_samples]
        
                                amis.append(adjusted_mutual_info_score(pred1_common, pred2_common)), aris.append(
                                    adjusted_rand_score(pred1_common, pred2_common))
        
                            alg_uns_metrics.loc[(alg_uns_metrics["dataset"] == dataset) &
                                                (alg_uns_metrics["missing_percentage"] == missing_percentage) &
                                                (alg_uns_metrics["amputation_mechanism"] == amputation_mechanism) &
                                                (alg_uns_metrics["algorithm"] == alg) & 
                                                (alg_uns_metrics["view_combination"] == view) & 
                                                (alg_uns_metrics["n_clusters"] == cluster),
                            ["AMI", "ARI"]] = [np.mean(amis), np.mean(aris)]

    return alg_uns_metrics

In [ ]:
# Function to normalise metrics with respect to a variable
def add_normalised_metric(df, variable_to_normalise, metric, greater_is_better=True):
    possible_variables = ["dataset", "algorithm", "missing_percentage", "amputation_mechanism", "view_combination", "n_clusters"]
    valid_variables = [var for var in possible_variables if var != variable_to_normalise]
    def recursive_loop(subset, remaining_vars, current_filters):
        if not remaining_vars:
            scores = subset[metric].values
            relative_score = scores / scores.max()
            if not greater_is_better:
                relative_score = 1 - relative_score
            condition = True
            for key, value in current_filters.items():
                condition &= (df[key] == value)
            df.loc[condition, f'normalised_{metric}'] = relative_score
            return
        current_var = remaining_vars[0]
        for unique_value in subset[current_var].unique():
            filtered_subset = subset[subset[current_var] == unique_value]
            recursive_loop(filtered_subset, remaining_vars[1:], {**current_filters, current_var: unique_value})
    df[f'normalised_{metric}'] = float('nan')
    recursive_loop(df, valid_variables, {})
    return df

In [ ]:
RANDOM_STATE = 42
n_clusters = 2

# Final clusters

To obtain the final clusters, the combination of miRNA and copy number (CNA) was used, and the algorithm MOFA applied. First, let's look at the stability of the clusters after n runs:

In [ ]:
import warnings
warnings.filterwarnings("ignore")
final_clusters = pd.read_csv('benchmarking_files/final_clusters_file.csv',
                             dtype={'view_combination': str},
                             converters={'y_pred': eval, 'y_pred_idx': eval, 
                                         'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

First, let's check if there are any very small clusters formed.

In [ ]:
valid_results, outlier_results, df_top_outliers = remove_small_clusters(final_clusters, 10, verbose=True)

In this case, there are no patients grouped into a small cluster.

In [ ]:
clin_results = clinical_enrichment(final_clusters, clinical_data_file)

In [ ]:
clin_results['pvalue_logrank'].mean()

In [ ]:
stability_dict = {}
for i in np.arange(10):
    subset = final_clusters.iloc[:(2**(i+1))]
    summarised_results = summarise_results(subset, random_state=RANDOM_STATE, progress_bar=True)
    stability_dict[i] = summarised_results['AMI'].iloc[0]

In [ ]:
stability_df = pd.DataFrame(stability_dict.items(), columns=['log2(n_runs)', 'AMI'])
stability_df['log2(n_runs)'] += 1
plt.figure(figsize=(10, 5))
sns.lineplot(x = stability_df['log2(n_runs)'], y = stability_df['AMI'])
plt.ylabel('Adjusted Mutual Information (AMI) index')
plt.ylim(0, 1)
plt.xlabel('log2(no. runs)')
plt.grid(alpha=0.4)
plt.savefig('figures/stability_finalclusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
consensus_matrix = consensus_matrix(final_clusters)

In [ ]:
# Applying k-means clustering to consensus matrix
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE).fit(consensus_matrix)
kmeans_df = pd.DataFrame({
    'Patient ID': consensus_matrix.index,
    'Cluster': kmeans.labels_})
kmeans_clusters = {}
for cluster in range(n_clusters):
    kmeans_clusters[f'Cluster_{cluster}'] = kmeans_df[kmeans_df['Cluster'] == cluster]

In [ ]:
# Applying hierarchical clustering to consensus matrix
cm = sns.clustermap(consensus_matrix, cmap='Blues', yticklabels=False, xticklabels=False, cbar_pos=None, dendrogram_ratio=0.1)
linkage_matrix = cm.dendrogram_row.linkage
cluster_assignments = fcluster(linkage_matrix, t=n_clusters, criterion='maxclust')
plt.savefig('figures/consensus_final_clusters.svg', bbox_inches='tight')
plt.show()

patient_ids = consensus_matrix.index
hierarchical_clusters = {}
for cluster in range(1, n_clusters + 1):
    hierarchical_clusters[f'Cluster_{cluster}'] = [patient_ids[i] for i, assignment in enumerate(cluster_assignments) if assignment == cluster]

In [ ]:
# Function to compare hierarchical clusters with K-means clusters
def compare_clusters(hierarchical_clusters, kmeans_clusters):
    hierarchical_sets = {key: set(values) for key, values in hierarchical_clusters.items()}
    kmeans_sets = {key: set(kmeans_clusters[key]['Patient ID'].tolist()) for key in kmeans_clusters}
    for h_key, h_set in hierarchical_sets.items():
        match_found = False
        for k_key, k_set in kmeans_sets.items():
            if h_set == k_set:
                print(f"Hierarchical {h_key} matches K-means {k_key}")
                match_found = True
                break
        if not match_found:
            print(f"Hierarchical {h_key} does not match any K-means cluster")
compare_clusters(hierarchical_clusters, kmeans_clusters)

# Clinical analysis

In [ ]:
clinical_data = clinical_data_file[['Patient ID', 'Mutation Count', 'Fraction Genome Altered', 'Diagnosis Age', 'Sex', 'Race Category', 
                                    'Adjuvant Postoperative Targeted Therapy Administered Indicator', 'Alcohol History Documented', 'Tumor resected max dimension',
                                    'American Joint Committee on Cancer Metastasis Stage Code', 'American Joint Committee on Cancer Tumor Stage Code',
                                    'Chronic Pancreatitis Personal Medical History Indicator', 'Did patient start adjuvant postoperative radiotherapy?', 
                                    'Disease Free Status', 'Family History of Cancer', 'Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code', 
                                    'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Neoplasm Histologic Grade',
                                    'New Neoplasm Event Post Initial Therapy Indicator', 'Overall Survival (Months)', 'Overall Survival Status', 'Disease Free (Months)', 
                                    'Participant Personal Medical History Diabetes Mellitus Ind-3', 'Patient Primary Tumor Site', 'Prior Cancer Diagnosis Occurence', 
                                    'Surgical Margin Resection Status', 'Patient Smoking History Category', 'Person Neoplasm Status', 'Primary Therapy Outcome Success Type']]
clinical_data['Overall Survival Status'] = clinical_data['Overall Survival Status'].str.split(':').str[0].astype(int)
clinical_data['Patient Smoking History Category'] = (clinical_data['Patient Smoking History Category']
                                                         .where(clinical_data['Patient Smoking History Category'].isna(), 
                                                                clinical_data['Patient Smoking History Category'].astype(float).astype(str)))
clinical_data.set_index('Patient ID', inplace=True)
clinical_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1]) - 1
    clinical_data.loc[clinical_data.index.isin(patients), 'Cluster'] = cluster_label
clinical_data.dropna(subset=['Cluster'], inplace=True)

### Survival curves

In [ ]:
# Check if weights are needed
from scipy.stats import linregress
test_weights_survival = clinical_data[['Overall Survival Status', 'Cluster']]
test_weights_survival['Cluster'] = pd.to_numeric(test_weights_survival['Cluster'])
result_weights_survival = linregress(x=test_weights_survival['Overall Survival Status'].values, y=test_weights_survival['Cluster'].values)
print(f"p = {result_weights_survival.pvalue}")

Since the p-value is < 0.05, we can assume that the censoring isn't random, hence weights are needed

In [ ]:
survival_df = clinical_data[['Overall Survival Status', 'Overall Survival (Months)', 'Cluster']]
survival_df['Weights'] = None
for i in np.arange(min(survival_df['Overall Survival (Months)']), max(survival_df['Overall Survival (Months)']), 6):
    interval_df = survival_df[(survival_df['Overall Survival (Months)'] >= i) & (survival_df['Overall Survival (Months)'] < i + 6)].copy()
    censored_data = interval_df['Overall Survival Status'].to_numpy()
    unique_classes = np.unique(censored_data)
    if censored_data.size != 0:
        class_weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=censored_data)
        weight_mapping = dict(zip(unique_classes, class_weights))
        interval_df['Weights'] = interval_df['Overall Survival Status'].map(weight_mapping)
        survival_df.update(interval_df[['Weights']])
survival_df['Weights'] = pd.to_numeric(survival_df['Weights'])
survival_df.sort_values(by='Cluster', ascending=True, inplace=True)    # order to have analysis with respect to cluster 0

In [ ]:
colorblind_palette = sns.color_palette('colorblind')
plt.figure(figsize=(15, 7))
kmf_list = []
event_dfs = []
for cluster in survival_df['Cluster'].unique():
    cluster_data = survival_df[survival_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Overall Survival (Months)'], cluster_data['Overall Survival Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(show_censors=True, color=colorblind_palette[cluster])
    kmf_list.append(kmf)
    event_table = kmf.event_table
    df = pd.DataFrame({'at_risk': event_table['at_risk'].round(0).astype(int), 
                       'censored': event_table['censored'].round(0).astype(int), 
                       'events': event_table['observed'].round(0).astype(int)})
    event_dfs.append(df)
plt.text(x=0.1, y=0.25, s='Log-rank, p = 0.113')    # from R-code
plt.xlabel('Time (months)')
max_time = max(survival_df['Overall Survival (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.ylabel('Survival probability')
add_at_risk_counts(*kmf_list, ax=plt.gca())

cph_data = survival_df.reset_index(drop=True)
cph = CoxPHFitter()
cph_df = cph.fit(cph_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status', weights_col='Weights').summary
plt.text(x=0.1, y=0.15, s=(f"HR (weights) = {cph_df['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_df['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_df['exp(coef) upper 95%'].loc['Cluster']:.2f})"))
cph = CoxPHFitter()
cph_df = cph.fit(cph_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
plt.text(x=0.1, y=0.05, s=(f"HR (no weights) = {cph_df['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_df['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_df['exp(coef) upper 95%'].loc['Cluster']:.2f})"))

plt.savefig('figures/logrank_final_clusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
time_intervals = np.arange(0, 80, 6)
fisher_dfs = {}
i=0
for df in event_dfs:
    df['cumulative_censored'] = df['censored'].cumsum()
    df['cumulative_events'] = df['events'].cumsum()
    df_interpolated = df.reindex(time_intervals, method='nearest').interpolate(method='index')
    result_df = pd.DataFrame({
        'at risk': df_interpolated['at_risk'].astype(int),
        'censored': df_interpolated['cumulative_censored'].cumsum().astype(int),
        'events': df_interpolated['cumulative_events'].cumsum().astype(int)}).T
    result_df.columns = time_intervals
    fisher_dfs[i] = result_df
    i += 1

In [ ]:
colorblind_palette = sns.color_palette('colorblind')
# plt.rcParams.update({'text.usetex': False, "svg.fonttype": 'none'})
plt.figure(figsize=(15, 7))
kmf_list = []
event_dfs = []
for cluster in survival_df['Cluster'].unique():
    cluster_data = survival_df[survival_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Overall Survival (Months)'], cluster_data['Overall Survival Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(color=colorblind_palette[cluster], ci_show=False)
    kmf_list.append(kmf)
    event_table = kmf.event_table
    df = pd.DataFrame({'at_risk': event_table['at_risk'].round(0).astype(int), 
                       'censored': event_table['censored'].round(0).astype(int), 
                       'events': event_table['observed'].round(0).astype(int)})
    event_dfs.append(df)
plt.xlabel('Time (months)')
max_time = max(survival_df['Overall Survival (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.ylabel('Survival probability')
plt.savefig('figures/rmst_survival.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Cox Proportional hazards, Fisher's test and RMST results at 6, 12, 36 and 60 months
months = [6, 12, 18, 24, 36, 60]
print('Hazard ratios with weights')
for time in months: 
    print(f"{time} months:")
    data_surv = survival_df.copy()
    time_subset = survival_df['Overall Survival (Months)'] <= time
    data_surv.loc[~time_subset, 'Overall Survival Status'] = 0
    # Cox proportional hazards model
    cph_data_surv = data_surv.reset_index(drop=True)
    cph = CoxPHFitter()
    cph_surv = cph.fit(cph_data_surv, duration_col='Overall Survival (Months)', event_col='Overall Survival Status', weights_col='Weights').summary
    print(f"Hazard Ratio = {cph_surv['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_surv['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_surv['exp(coef) upper 95%'].loc['Cluster']:.2f})")

print('\nHazard ratios without weights')
for time in months: 
    print(f"{time} months:")
    data_surv = survival_df.copy()
    time_subset = survival_df['Overall Survival (Months)'] <= time
    data_surv.loc[~time_subset, 'Overall Survival Status'] = 0
    # Cox proportional hazards model
    cph_data_surv = data_surv.reset_index(drop=True)
    cph = CoxPHFitter()
    cph_surv = cph.fit(cph_data_surv, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    print(f"Hazard Ratio = {cph_surv['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_surv['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_surv['exp(coef) upper 95%'].loc['Cluster']:.2f})")

### Disease free curves

In [ ]:
test_weights_disease_free = clinical_data[['Disease Free Status', 'Cluster']]
test_weights_disease_free = test_weights_disease_free.dropna(how='any',axis=0)
test_weights_disease_free['Cluster'] = pd.to_numeric(test_weights_disease_free['Cluster'])
test_weights_disease_free['Disease Free Status'] = test_weights_disease_free['Disease Free Status'].str.split(':').str[0].astype(int)
result_weights_disease_free = linregress(x=test_weights_disease_free['Disease Free Status'].values, y=test_weights_disease_free['Cluster'].values)
print(f"p = {result_weights_disease_free.pvalue}")

Since the p-value is > 0.05, we have insufficient evidence to reject the null hypothesis, hence assume the censoring was random, and don't need weights for future calculations.

In [ ]:
disease_free_df = clinical_data[['Disease Free Status', 'Disease Free (Months)', 'Cluster']]
disease_free_df.sort_values(by='Cluster', ascending=True, inplace=True)    # order to have analysis with respect to cluster 0
disease_free_df = disease_free_df.dropna(how='any',axis=0)
disease_free_df['Disease Free Status'] = disease_free_df['Disease Free Status'].str.split(':').str[0].astype(int)

From the R code, we can see that the data for tumor recurrence does not follow the proportional hazards model. Hence, we need to stratify the clusters so that we can use the assumption of proportional hazards. 

In [ ]:
kmf_list = []
plt.figure(figsize=(15, 7))
for cluster in sorted(disease_free_df['Cluster'].unique()):
    cluster_data = disease_free_df[disease_free_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Disease Free (Months)'], cluster_data['Disease Free Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(show_censors=True, color=colorblind_palette[cluster])
    kmf_list.append(kmf)
max_time= max(disease_free_df['Disease Free (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.text(x=0.1, y=0.15, s= 'Log-rank, p = 0.216')    # from R-code
plt.xlabel('Months')
plt.ylabel('Disease free probability')
add_at_risk_counts(*kmf_list, ax=plt.gca())

# CPH model
cph_data = disease_free_df.reset_index(drop=True)
cph = CoxPHFitter()
cph_df = cph.fit(cph_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
plt.text(x=0.1, y=0.05, s=(f"HR = {cph_df['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_df['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_df['exp(coef) upper 95%'].loc['Cluster']:.2f})"))

plt.savefig('figures/logrank_disease_free_clusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
kmf_list = []
plt.figure(figsize=(15, 7))
for cluster in sorted(disease_free_df['Cluster'].unique()):
    cluster_data = disease_free_df[disease_free_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Disease Free (Months)'], cluster_data['Disease Free Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(color=colorblind_palette[cluster], ci_show=False)
    kmf_list.append(kmf)
max_time= max(disease_free_df['Disease Free (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.xlabel('Time (months)')
plt.ylabel('Disease free probability')
plt.savefig('figures/rmst_diseasefree.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Cox Proportional hazards, Fisher's test and RMST results at 6, 12, 36 and 60 months
months = [6, 12, 18, 24, 36, 60]
for time in months: 
    print(f"{time} months:")
    data_diseasefree = disease_free_df.copy()
    time_subset = disease_free_df['Disease Free (Months)'] <= time
    data_diseasefree.loc[~time_subset, 'Disease Free Status'] = 0
    # Cox proportional hazards model
    cph_data_diseasefree = data_diseasefree.reset_index(drop=True)
    cph = CoxPHFitter()
    cph_diseasefree = cph.fit(cph_data_diseasefree, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    print(f"Hazard Ratio = {cph_diseasefree['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_diseasefree['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_diseasefree['exp(coef) upper 95%'].loc['Cluster']:.2f})")

### Enrichment of clinical labels

In [ ]:
# Enrichment of clinical labels
clinical_labels = clinical_data.copy()
clinical_enrichment = {}
for variable in clinical_labels.columns:
    if pd.api.types.is_numeric_dtype(clinical_labels[variable]):
        test_numerical = [
            clinical_labels[clinical_labels['Cluster'] == cluster][variable].dropna().to_numpy() 
            for cluster in clinical_labels['Cluster'].unique()]
        stat, p_value_kruskal = kruskal(*test_numerical)
        clinical_enrichment[variable] = p_value_kruskal
    else: 
        test_discrete = pd.crosstab(clinical_labels['Cluster'], clinical_labels[variable])
        chi2, p_value_chi2, dof, expected = chi2_contingency(test_discrete)
        clinical_enrichment[variable] = p_value_chi2
del clinical_enrichment['Cluster'], clinical_enrichment['Overall Survival (Months)'], clinical_enrichment['Overall Survival Status']
clinical_enrichment_pvalues = pd.DataFrame.from_dict(clinical_enrichment, orient='index', columns=['Original p-value'])
reject, pvals_corr, asidack, abonf = statsmodels.stats.multitest.multipletests(pvals=clinical_enrichment_pvalues['Original p-value'], alpha=0.05, 
                                                                               method='fdr_bh', maxiter=1, is_sorted=False, returnsorted=False)
clinical_enrichment_pvalues['Adjusted p-value'] = pvals_corr
clinical_enrichment_pvalues['Significance'] = clinical_enrichment_pvalues['Adjusted p-value'].apply(lambda x: '*' if x < 0.05 else '')
clinical_enrichment_pvalues[clinical_enrichment_pvalues['Significance'] == '*']

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=clinical_data, x='Cluster', y='Mutation Count', showmeans=True, palette='colorblind', ax=ax[0])
ax[0].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['Mutation Count']:.3e}", ha='center', va='top', transform=ax[0].transAxes)

sns.boxplot(data=clinical_data, x='Cluster', y='Fraction Genome Altered', showmeans=True, palette='colorblind', ax=ax[1])
ax[1].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['Fraction Genome Altered']:.3e}", ha='center', va='top', transform=ax[1].transAxes)
ax[1].set_ylabel('Fraction Genome Altered (%)')

plt.tight_layout()
plt.show()

There seems to be an extreme outlier in mutation count, so let's repeat the test after removing the outlier patient.

In [ ]:
outlier = clinical_data[clinical_data['Mutation Count'] > 500].index    # isolate the outlier from the plots
clinical_data_no_outlier = clinical_data.drop(outlier)
clinical_enrichment_no_outlier = {}
for variable in clinical_data_no_outlier.columns:
    if pd.api.types.is_numeric_dtype(clinical_data_no_outlier[variable]):
        test_numerical = [
            clinical_data_no_outlier[clinical_data_no_outlier['Cluster'] == cluster][variable].dropna().to_numpy() 
            for cluster in clinical_data_no_outlier['Cluster'].unique()]
        stat, p_value_kruskal = kruskal(*test_numerical)
        clinical_enrichment_no_outlier[variable] = p_value_kruskal
    else: 
        test_discrete = pd.crosstab(clinical_data_no_outlier['Cluster'], clinical_data_no_outlier[variable])
        chi2, p_value_chi2, dof, expected = chi2_contingency(test_discrete)
        clinical_enrichment_no_outlier[variable] = p_value_chi2
del clinical_enrichment_no_outlier['Cluster'], clinical_enrichment_no_outlier['Overall Survival (Months)'], clinical_enrichment_no_outlier['Overall Survival Status']
clinical_enrichment_pvalues_no_outlier = pd.DataFrame.from_dict(clinical_enrichment_no_outlier, orient='index', columns=['Original p-value'])
reject, pvals_corr, asidack, abonf = statsmodels.stats.multitest.multipletests(pvals=clinical_enrichment_pvalues_no_outlier['Original p-value'], alpha=0.05, 
                                                                               method='fdr_bh', maxiter=1, is_sorted=False, returnsorted=False)
clinical_enrichment_pvalues_no_outlier['Adjusted p-value'] = pvals_corr
clinical_enrichment_pvalues_no_outlier['Significance'] = clinical_enrichment_pvalues_no_outlier['Adjusted p-value'].apply(lambda x: '*' if x < 0.05 else '')
clinical_enrichment_pvalues_no_outlier[clinical_enrichment_pvalues_no_outlier['Significance'] == '*']

In [ ]:
from statannotations.Annotator import Annotator
pairs = [(0, 1)]
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='Mutation Count', showmeans=True, palette='colorblind', ax=ax[0])
p_value_mutcount = clinical_enrichment_pvalues['Adjusted p-value'].loc['Mutation Count']
annotator_mutcount = Annotator(ax[0], pairs, data=clinical_data_no_outlier, x='Cluster', y='Mutation Count')
annotator_mutcount.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_mutcount.set_custom_annotations([f"p = {p_value_mutcount:.2e}"])
annotator_mutcount.annotate()
ax[0].set_ylabel('Mutation Count')

sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='Fraction Genome Altered', showmeans=True, palette='colorblind', ax=ax[1])
p_value_fga = clinical_enrichment_pvalues['Adjusted p-value'].loc['Fraction Genome Altered']
annotator_fga = Annotator(ax[1], pairs, data=clinical_data_no_outlier, x='Cluster', y='Fraction Genome Altered')
annotator_fga.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_fga.set_custom_annotations([f"p = {p_value_fga:.2e}"])
annotator_fga.annotate()
ax[1].set_ylabel('Fraction Genome Altered (%)')

plt.tight_layout()
plt.savefig('figures/enriched_labels_boxplots.svg', bbox_inches='tight')
plt.show()

### Minimal biomarker panel

From the previous results, there are no differences in survival. However, the clusters are always very consistent, which could indicate that there is some underlying information that might be helpful or an indicator for pancreatic cancer. For that reason, we will check for biomarkers using a cross-validation and feature selection approach. Four methods will be used.

In [ ]:
# Get data files required
cna_file = pd.read_csv('processed_data/CNA_processed.csv', index_col=0)
methylation_file = pd.read_csv('processed_data/methylation_processed.csv', index_col=0)
complete_data = pd.merge(cna_file, methylation_file, how='inner', left_index=True, right_index=True)
patients_data = clinical_data[['Overall Survival Status', 'Overall Survival (Months)']]

In [ ]:
complete_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1]) - 1
    complete_data.loc[complete_data.index.isin(patients), 'Cluster'] = cluster_label
complete_data.dropna(subset=['Cluster'], inplace=True)

In [ ]:
X = complete_data.iloc[:, :-1].to_numpy(dtype=float)
y = complete_data.iloc[:, -1].to_numpy(dtype=int)
feature_names = complete_data.iloc[:, :-1].columns.to_numpy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

**1. Random Forest Classifier (RFC)**

In [ ]:
crossval_rfc = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE), X, y, 
                               cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.2f, standard deviation: %0.2f" % (crossval_rfc.mean(), crossval_rfc.std()))

In [ ]:
pipeline_rfc = make_pipeline(SequentialFeatureSelector(RandomForestClassifier(random_state=RANDOM_STATE), tol=0.01, n_jobs=6,
                                                      cv=skf, scoring='matthews_corrcoef', direction='forward'), 
                            RandomForestClassifier(random_state=RANDOM_STATE, oob_score=True))
scores_rfc = cross_val_score(pipeline_rfc, X, y, cv=skf, scoring='matthews_corrcoef')
pipeline_rfc.fit(X, y)
biomarkers_rfc = feature_names[pipeline_rfc.named_steps['sequentialfeatureselector'].get_support()].tolist()
print(f"Features selected: {biomarkers_rfc}")
print(f"Matthews Correlation Coefficient: {scores_rfc.mean():.3f}, standard deviation: {scores_rfc.std():.3f}")
print(f"Out-of-bag (OOB) score: {pipeline_rfc.named_steps['randomforestclassifier'].oob_score_}")

In [ ]:
biomarkers_data_rfc = complete_data[biomarkers_rfc]
patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)']]
biomarkers_survival_rfc = biomarkers_data_rfc.merge(patients_data, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 2, figsize=(20, 4), width_ratios=[0.3, 0.7])
cph = CoxPHFitter()
rfc_biomarkers_cph = cph.fit(biomarkers_survival_rfc, 'Overall Survival (Months)', 'Overall Survival Status').summary
rfc_biomarkers_cph.reset_index(inplace=True)
fp.forestplot(rfc_biomarkers_cph, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-3, 3.5, 1), ax=ax[0])

boxplots_df_rfc = biomarkers_survival_rfc.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_rfc_melted = boxplots_df_rfc.melt(id_vars="Cluster", value_vars=biomarkers_rfc,
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_rfc_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
ax[1].set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig('figures/biomarkers_rfc.svg', bbox_inches='tight')
plt.show()

for biomarker in biomarkers_rfc:
    group0 = boxplots_df_rfc[boxplots_df_rfc['Cluster'] == 0][biomarker]
    group1 = boxplots_df_rfc[boxplots_df_rfc['Cluster'] == 1][biomarker]
    stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{biomarker} p-value: {p_value}")

In [ ]:
biomarkers_data_rfc = complete_data[biomarkers_rfc]
patients_data_diseasefree = disease_free_df[['Disease Free Status', 'Disease Free (Months)']]
biomarkers_diseasefree_rfc = biomarkers_data_rfc.merge(patients_data_diseasefree, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 2, figsize=(20, 4), width_ratios=[0.3, 0.7])
cph = CoxPHFitter()
rfc_biomarkers_cph_diseasefree = cph.fit(biomarkers_diseasefree_rfc, 'Disease Free (Months)', 'Disease Free Status').summary
rfc_biomarkers_cph_diseasefree.reset_index(inplace=True)
fp.forestplot(rfc_biomarkers_cph_diseasefree, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-5, 7, 1), ax=ax[0])

boxplots_df_rfc_diseasefree = biomarkers_diseasefree_rfc.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_rfc_melted_diseasefree = boxplots_df_rfc_diseasefree.melt(id_vars="Cluster", value_vars=biomarkers_rfc,
                                                                      var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_rfc_melted_diseasefree, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
ax[1].set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig('figures/biomarkers_rfc_diseasefree.svg', bbox_inches='tight')
plt.show()

for biomarker in biomarkers_rfc:
    group0 = boxplots_df_rfc_diseasefree[boxplots_df_rfc_diseasefree['Cluster'] == 0][biomarker]
    group1 = boxplots_df_rfc_diseasefree[boxplots_df_rfc_diseasefree['Cluster'] == 1][biomarker]
    stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{biomarker} p-value: {p_value}")

In [ ]:
complete_data_rfc = complete_data.drop(columns=biomarkers_rfc)
X_rfc = complete_data_rfc.iloc[:, :-1].to_numpy(dtype=float)
y_rfc = complete_data_rfc.iloc[:, -1].to_numpy(dtype=int)
feature_names_rfc = complete_data_rfc.iloc[:, :-1].columns.to_numpy()

print('Cross validation:')
new_crossval_rfc = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE), X_rfc, y_rfc, 
                                   cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.2f, standard deviation: %0.2f" % (new_crossval_rfc.mean(), new_crossval_rfc.std()))

print('Feature selection:')
new_pipeline_rfc = make_pipeline(SequentialFeatureSelector(RandomForestClassifier(random_state=RANDOM_STATE), tol=0.01, n_jobs=6,
                                                           cv=skf, scoring='matthews_corrcoef', direction='forward'), 
                                 RandomForestClassifier(random_state=RANDOM_STATE, oob_score=True))
new_scores_rfc = cross_val_score(new_pipeline_rfc, X_rfc, y_rfc, cv=skf, scoring='matthews_corrcoef')
new_pipeline_rfc.fit(X_rfc, y_rfc)
features_selected_rfc = feature_names_rfc[new_pipeline_rfc.named_steps['sequentialfeatureselector'].get_support()].tolist()
print(f"Features selected: {features_selected_rfc}")
print(f"Matthews Correlation Coefficient: {new_scores_rfc.mean():.3f}, standard deviation: {new_scores_rfc.std():.3f}")
print(f"Out-of-bag (OOB) score: {new_pipeline_rfc.named_steps['randomforestclassifier'].oob_score_}")

**2. Estimator (KBinsDiscretizer + RFC)**

In [ ]:
estimator = make_pipeline(KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', random_state=RANDOM_STATE), 
                          RandomForestClassifier(random_state=RANDOM_STATE))
crossval_est = cross_val_score(estimator, X, y, cv=skf, scoring = 'matthews_corrcoef')
print("MCC score: %0.2f, standard deviation: %0.2f" % (crossval_est.mean(), crossval_est.std()))

In [ ]:
estimator1 = make_pipeline(KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', random_state=RANDOM_STATE), 
                          RandomForestClassifier(random_state=RANDOM_STATE))
pipeline_est = make_pipeline(SequentialFeatureSelector(estimator1, tol=0.01, cv=skf, scoring='matthews_corrcoef', direction='forward', n_jobs=6), 
                             RandomForestClassifier(random_state=RANDOM_STATE, oob_score=True))
scores_est = cross_val_score(pipeline_est, X, y, cv=skf, scoring='matthews_corrcoef')
pipeline_est.fit(X, y)
biomarkers_est = feature_names[pipeline_est.named_steps['sequentialfeatureselector'].get_support()].tolist()
print(f"Features selected: {biomarkers_est}")
print(f"Matthews Correlation Coefficient: {scores_est.mean():.3f}, standard deviation: {scores_est.std():.3f}")
print(f"Out-of-bag (OOB) score: {pipeline_est.named_steps['randomforestclassifier'].oob_score_}")

In [ ]:
biomarkers_data_est = complete_data[biomarkers_est]
patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)']]
biomarkers_survival_est = biomarkers_data_est.merge(patients_data, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 3, figsize=(20, 4))
cph_est = CoxPHFitter()
biomarkers_cph_est = cph_est.fit(biomarkers_survival_est, 'Overall Survival (Months)', 'Overall Survival Status').summary
biomarkers_cph_est.reset_index(inplace=True)
fp.forestplot(biomarkers_cph_est, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-1, 4, 1), ax=ax[0])
boxplots_df_est = biomarkers_survival_est.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_est_melted = boxplots_df_est.melt(id_vars="Cluster", value_vars=['cg15920654', 'cg24708471'],
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_est_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
handles, labels = ax[1].get_legend_handles_labels()
ax[1].legend(handles[:2], labels, title='Cluster', loc='upper left')

contingency_table_est = pd.crosstab(boxplots_df_est['21q11.2'], boxplots_df_est['Cluster'])
sns.heatmap(contingency_table_est, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2])
ax[2].set_ylabel('21q11.2 expression')
ax[2].legend([]).remove()

plt.tight_layout()
plt.savefig('figures/biomarkers_est.svg', bbox_inches='tight')
plt.show()

In [ ]:
biomarkers_data_est = complete_data[biomarkers_est]
patients_data_diseasefree = disease_free_df[['Disease Free Status', 'Disease Free (Months)']]
biomarkers_diseasefree_est = biomarkers_data_est.merge(patients_data_diseasefree, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 3, figsize=(20, 4))
cph = CoxPHFitter()
est_biomarkers_cph_diseasefree = cph.fit(biomarkers_diseasefree_est, 'Disease Free (Months)', 'Disease Free Status').summary
est_biomarkers_cph_diseasefree.reset_index(inplace=True)
fp.forestplot(est_biomarkers_cph_diseasefree, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-5, 7, 1), ax=ax[0])

boxplots_df_est_diseasefree = biomarkers_diseasefree_est.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_est_melted_diseasefree = boxplots_df_est_diseasefree.melt(id_vars="Cluster", value_vars=['cg15920654', 'cg24708471'],
                                                                      var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_est_melted_diseasefree, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
# ax[1].set_ylim(0, 1.2)
handles, labels = ax[1].get_legend_handles_labels()
ax[1].legend(handles[:2], labels, title='Cluster', loc='upper left')

contingency_table_est = pd.crosstab(boxplots_df_est_diseasefree['21q11.2'], boxplots_df_est_diseasefree['Cluster'])
sns.heatmap(contingency_table_est, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2])
ax[2].set_ylabel('21q11.2 expression')
ax[2].legend([]).remove()

plt.tight_layout()
plt.savefig('figures/biomarkers_est_diseasefree.svg', bbox_inches='tight')
plt.show()

for biomarker in ['cg15920654', 'cg24708471']:
    group0 = boxplots_df_est_diseasefree[boxplots_df_est_diseasefree['Cluster'] == 0][biomarker]
    group1 = boxplots_df_est_diseasefree[boxplots_df_est_diseasefree['Cluster'] == 1][biomarker]
    stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{biomarker} p-value: {p_value}")
stat, pval = fisher_exact(contingency_table_est)
print(f"21q11.2 p-value: {pval}")

In [ ]:
biomarkers_data_est = complete_data[biomarkers_est]
patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)']]
biomarkers_survival_est = biomarkers_data_est.merge(patients_data, left_index=True, right_index=True)
biomarkers_survival_est = biomarkers_survival_est[biomarkers_survival_est['Overall Survival (Months)'] < 80]

fig, ax= plt.subplots(1, 3, figsize=(20, 4))
cph_est = CoxPHFitter()
biomarkers_cph_est = cph_est.fit(biomarkers_survival_est, 'Overall Survival (Months)', 'Overall Survival Status').summary
biomarkers_cph_est.reset_index(inplace=True)
fp.forestplot(biomarkers_cph_est, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-1, 4, 1), ax=ax[0])
boxplots_df_est = biomarkers_survival_est.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_est_melted = boxplots_df_est.melt(id_vars="Cluster", value_vars=['cg15920654', 'cg24708471'],
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_est_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
handles, labels = ax[1].get_legend_handles_labels()
ax[1].legend(handles[:2], labels, title='Cluster', loc='upper left')

contingency_table_est = pd.crosstab(boxplots_df_est['21q11.2'], boxplots_df_est['Cluster'])
sns.heatmap(contingency_table_est, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2])
ax[2].set_ylabel('21q11.2 expression')
ax[2].legend([]).remove()

plt.tight_layout()
plt.show()

In [ ]:
biomarkers_survival_est = biomarkers_survival_est[biomarkers_survival_est['Overall Survival (Months)'] < 60]

fig, ax= plt.subplots(1, 3, figsize=(20, 4))
cph_est = CoxPHFitter()
biomarkers_cph_est = cph_est.fit(biomarkers_survival_est, 'Overall Survival (Months)', 'Overall Survival Status').summary
biomarkers_cph_est.reset_index(inplace=True)
fp.forestplot(biomarkers_cph_est, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-1, 4, 1), ax=ax[0])
boxplots_df_est = biomarkers_survival_est.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_est_melted = boxplots_df_est.melt(id_vars="Cluster", value_vars=['cg15920654', 'cg24708471'],
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_est_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
handles, labels = ax[1].get_legend_handles_labels()
ax[1].legend(handles[:2], labels, title='Cluster', loc='upper left')

contingency_table_est = pd.crosstab(boxplots_df_est['21q11.2'], boxplots_df_est['Cluster'])
sns.heatmap(contingency_table_est, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2])
ax[2].set_ylabel('21q11.2 expression')
ax[2].legend([]).remove()

plt.tight_layout()
plt.savefig('figures/biomarkers_y5.svg', bbox_inches='tight')
plt.show()

In [ ]:
biomarkers_survival_est = biomarkers_survival_est[biomarkers_survival_est['Overall Survival (Months)'] < 36]

fig, ax= plt.subplots(1, 3, figsize=(20, 4))
cph_est = CoxPHFitter()
biomarkers_cph_est = cph_est.fit(biomarkers_survival_est, 'Overall Survival (Months)', 'Overall Survival Status').summary
biomarkers_cph_est.reset_index(inplace=True)
fp.forestplot(biomarkers_cph_est, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-1, 4, 1), ax=ax[0])
boxplots_df_est = biomarkers_survival_est.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_est_melted = boxplots_df_est.melt(id_vars="Cluster", value_vars=['cg15920654', 'cg24708471'],
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_est_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
handles, labels = ax[1].get_legend_handles_labels()
ax[1].legend(handles[:2], labels, title='Cluster', loc='upper left')

contingency_table_est = pd.crosstab(boxplots_df_est['21q11.2'], boxplots_df_est['Cluster'])
sns.heatmap(contingency_table_est, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2])
ax[2].set_ylabel('21q11.2 expression')
ax[2].legend([]).remove()

plt.tight_layout()
plt.savefig('figures/biomarkers_y3.svg', bbox_inches='tight')
plt.show()

In [ ]:
biomarkers_survival_est = biomarkers_survival_est[biomarkers_survival_est['Overall Survival (Months)'] < 12]

fig, ax= plt.subplots(1, 3, figsize=(20, 4))
cph_est = CoxPHFitter()
biomarkers_cph_est = cph_est.fit(biomarkers_survival_est, 'Overall Survival (Months)', 'Overall Survival Status').summary
biomarkers_cph_est.reset_index(inplace=True)
fp.forestplot(biomarkers_cph_est, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-1, 4, 1), ax=ax[0])
boxplots_df_est = biomarkers_survival_est.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_est_melted = boxplots_df_est.melt(id_vars="Cluster", value_vars=['cg15920654', 'cg24708471'],
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_est_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
handles, labels = ax[1].get_legend_handles_labels()
ax[1].legend(handles[:2], labels, title='Cluster', loc='upper left')

contingency_table_est = pd.crosstab(boxplots_df_est['21q11.2'], boxplots_df_est['Cluster'])
sns.heatmap(contingency_table_est, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2])
ax[2].set_ylabel('21q11.2 expression')
ax[2].legend([]).remove()

plt.tight_layout()
plt.savefig('figures/biomarkers_y1.svg', bbox_inches='tight')
plt.show()

In [ ]:
complete_data_est = complete_data.drop(columns=biomarkers_est)
X_est = complete_data_est.iloc[:, :-1].to_numpy(dtype=float)
y_est = complete_data_est.iloc[:, -1].to_numpy(dtype=int)
feature_names_est = complete_data_est.iloc[:, :-1].columns.to_numpy()

print('Cross validation:')
new_estimator = make_pipeline(KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', random_state=RANDOM_STATE), 
                              RandomForestClassifier(random_state=RANDOM_STATE))
new_crossval_est = cross_val_score(new_estimator, X_est, y_est, cv=skf, scoring = 'matthews_corrcoef')
print("MCC score: %0.2f, standard deviation: %0.2f" % (new_crossval_est.mean(), new_crossval_est.std()))

print('Feature selection:')
new_estimator1 = make_pipeline(KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', random_state=RANDOM_STATE), 
                               RandomForestClassifier(random_state=RANDOM_STATE))
new_pipeline_est = make_pipeline(SequentialFeatureSelector(new_estimator1, tol=0.01, cv=skf, scoring='matthews_corrcoef', direction='forward', n_jobs=4), 
                                 RandomForestClassifier(random_state=RANDOM_STATE, oob_score=True))
new_scores_est = cross_val_score(new_pipeline_est, X_est, y_est, cv=skf, scoring='matthews_corrcoef')
new_pipeline_est.fit(X_est, y_est)
features_selected_est = feature_names_est[new_pipeline_est.named_steps['sequentialfeatureselector'].get_support()].tolist()
print(f"Features selected: {features_selected_est}")
print(f"Matthews Correlation Coefficient: {new_scores_est.mean():.3f}, standard deviation: {new_scores_est.std():.3f}")
print(f"Out-of-bag (OOB) score: {new_pipeline_est.named_steps['randomforestclassifier'].oob_score_}")

**3. RFC with class_weight = 'balanced_subsample'**

In [ ]:
crossval_bw = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                              X, y, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.2f, standard deviation: %0.2f" % (crossval_bw.mean(), crossval_bw.std()))

In [ ]:
pipeline_bw = make_pipeline(SequentialFeatureSelector(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                                                      tol=0.01, n_jobs=6, cv=skf, scoring='matthews_corrcoef', direction='forward'), 
                            RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_bw = cross_val_score(pipeline_bw, X, y, cv=skf, scoring='matthews_corrcoef')
pipeline_bw.fit(X, y)
biomarkers_bw = feature_names[pipeline_bw.named_steps['sequentialfeatureselector'].get_support()].tolist()
print(f"Features selected: {biomarkers_bw}")
print(f"Matthews Correlation Coefficient: {scores_bw.mean():.3f}, standard deviation: {scores_bw.std():.3f}")
print(f"Out-of-bag (OOB) score: {pipeline_bw.named_steps['randomforestclassifier'].oob_score_}")

In [ ]:
biomarkers_data_bw = complete_data[biomarkers_bw]
patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)']]
biomarkers_survival_bw = biomarkers_data_bw.merge(patients_data, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 2, figsize=(20, 4), width_ratios=[0.3, 0.7])
cph = CoxPHFitter()
bw_biomarkers_cph = cph.fit(biomarkers_survival_bw, 'Overall Survival (Months)', 'Overall Survival Status').summary
bw_biomarkers_cph.reset_index(inplace=True)
fp.forestplot(bw_biomarkers_cph, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-3, 3.5, 1), ax=ax[0])

boxplots_df_bw = biomarkers_survival_bw.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_bw_melted = boxplots_df_bw.melt(id_vars="Cluster", value_vars=biomarkers_bw,
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_bw_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
ax[1].set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig('figures/biomarkers_bw.svg', bbox_inches='tight')
plt.show()

for biomarker in biomarkers_bw:
    group0 = boxplots_df_bw[boxplots_df_bw['Cluster'] == 0][biomarker]
    group1 = boxplots_df_bw[boxplots_df_bw['Cluster'] == 1][biomarker]
    stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{biomarker} p-value: {p_value}")

In [ ]:
biomarkers_data_bw = complete_data[biomarkers_bw]
patients_data_diseasefree = disease_free_df[['Disease Free Status', 'Disease Free (Months)']]
biomarkers_diseasefree_bw = biomarkers_data_bw.merge(patients_data_diseasefree, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 2, figsize=(20, 4), width_ratios=[0.3, 0.7])
cph = CoxPHFitter()
bw_biomarkers_cph_diseasefree = cph.fit(biomarkers_diseasefree_bw, 'Disease Free (Months)', 'Disease Free Status').summary
bw_biomarkers_cph_diseasefree.reset_index(inplace=True)
fp.forestplot(bw_biomarkers_cph_diseasefree, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-5, 7, 1), ax=ax[0])

boxplots_df_bw_diseasefree = biomarkers_diseasefree_bw.merge(complete_data['Cluster'], left_index=True, right_index=True)
boxplots_df_bw_melted_diseasefree = boxplots_df_bw_diseasefree.melt(id_vars="Cluster", value_vars=biomarkers_bw,
                                                                      var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_bw_melted_diseasefree, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
ax[1].set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig('figures/biomarkers_bw_diseasefree.svg', bbox_inches='tight')
plt.show()

for biomarker in biomarkers_bw:
    group0 = boxplots_df_bw_diseasefree[boxplots_df_bw_diseasefree['Cluster'] == 0][biomarker]
    group1 = boxplots_df_bw_diseasefree[boxplots_df_bw_diseasefree['Cluster'] == 1][biomarker]
    stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{biomarker} p-value: {p_value}")

In [ ]:
complete_data_bw = complete_data.drop(columns=biomarkers_bw)
X_bw = complete_data_bw.iloc[:, :-1].to_numpy(dtype=float)
y_bw = complete_data_bw.iloc[:, -1].to_numpy(dtype=int)
feature_names_bw = complete_data_bw.iloc[:, :-1].columns.to_numpy()

print('Cross validation:')
new_crossval_bw = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'),
                                   X_bw, y_bw, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.2f, standard deviation: %0.2f" % (new_crossval_bw.mean(), new_crossval_bw.std()))

print('Feature selection:')
new_pipeline_bw = make_pipeline(SequentialFeatureSelector(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                                                           tol=0.01, n_jobs=6, cv=skf, scoring='matthews_corrcoef', direction='forward'), 
                                 RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
new_scores_bw = cross_val_score(new_pipeline_bw, X_bw, y_bw, cv=skf, scoring='matthews_corrcoef')
new_pipeline_bw.fit(X_bw, y_bw)
features_selected_bw = feature_names_bw[new_pipeline_bw.named_steps['sequentialfeatureselector'].get_support()].tolist()
print(f"Features selected: {features_selected_bw}")
print(f"Matthews Correlation Coefficient: {new_scores_bw.mean():.3f}, standard deviation: {new_scores_bw.std():.3f}")
print(f"Out-of-bag (OOB) score: {new_pipeline_bw.named_steps['randomforestclassifier'].oob_score_}")

In [ ]:
all_biomarkers = list(set(biomarkers_rfc + biomarkers_est + biomarkers_bw))
fig, ax = plt.subplots(2, 4, figsize=(30, 6))
ax = ax.flatten()
for i, biomarker in enumerate(all_biomarkers):
    biomarkers_data = complete_data[[biomarker]]
    patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)']]
    biomarkers_survival = biomarkers_data.merge(patients_data, left_index=True, right_index=True)
    cph = CoxPHFitter()
    biomarkers_cph = cph.fit(biomarkers_survival, 'Overall Survival (Months)', 'Overall Survival Status').summary
    biomarkers_cph.reset_index(inplace=True)
    fp.forestplot(biomarkers_cph, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate',
                  pval='p', color_alt_rows=True, xticks=np.arange(-3, 3.5, 1), ax=ax[i])   
plt.subplots_adjust(wspace=3, hspace=0.5)
plt.show()

### Heatmaps for methylation and CNA data, with cluster assignments

In [ ]:
# Get data from both methylation and CNA
CNA_file = pd.read_csv('processed_data/CNA_processed.csv', index_col=0)
CNA_file['Cluster'] = None
CNA_file.loc[CNA_file.index.isin(hierarchical_cluster0), 'Cluster'] = 0
CNA_file.loc[CNA_file.index.isin(hierarchical_cluster1), 'Cluster'] = 1
CNA_data = CNA_file.sort_values(by='Cluster')

methylation_file = pd.read_csv('processed_data/methylation_processed.csv', index_col=0)
methylation_file['Cluster'] = None
methylation_file.loc[methylation_file.index.isin(hierarchical_cluster0), 'Cluster'] = 0
methylation_file.loc[methylation_file.index.isin(hierarchical_cluster1), 'Cluster'] = 1
methylation_data = methylation_file.sort_values(by='Cluster')

# Plot heatmaps
clusters_CNA = CNA_data['Cluster'].unique()
palette_CNA = sns.color_palette("colorblind", len(clusters_CNA))
color_mapping_CNA = dict(zip(clusters_CNA, palette_CNA))
row_colors_CNA = CNA_data['Cluster'].map(color_mapping_CNA)
sns.clustermap(data=CNA_data.drop(columns=['Cluster']), row_colors=row_colors_CNA, cmap='Blues', row_cluster=False)
plt.show()
clusters_methylation = methylation_data['Cluster'].unique()
palette_methylation = sns.color_palette("colorblind", len(clusters_methylation))
color_mapping_methylation = dict(zip(clusters_methylation, palette_methylation))
row_colors_methylation = methylation_data['Cluster'].map(color_mapping_methylation)
sns.clustermap(data=methylation_data.drop(columns=['Cluster']), row_colors=row_colors_methylation, cmap='Blues', row_cluster=False)
plt.show()